# Mai24_CMLOPS_Accidents_Cdc_#1_Analyse_data

## 1- Exploration des données

In [98]:
# Import des bibliothèques nécessaires au projet
import pandas as pd
import numpy as np
import warnings
import matplotlib.pyplot as plt
import plotly.express as px
from plotly.offline import init_notebook_mode, iplot

# Ignorer les avertissements
warnings.filterwarnings("ignore", category=pd.errors.DtypeWarning)

### 1.1 Données 'lieux'

In [99]:
df_lieux=pd.read_csv ('lieux_2005a2021.csv', index_col=0)
df_lieux.head()

,num_acc,catr,voie,v1,v2,circ,nbv,pr,pr1,vosp,prof,plan,lartpc,larrout,surf,infra,situ,env1,annee,vma
1,200500000001,3.0,41.0,0.0,B,2.0,2.0,1.0,430.0,0.0,1.0,1.0,0.0,63.0,1.0,0.0,1.0,0.0,2005,NaN
2,200500000002,2.0,41.0,0.0,NaN,0.0,2.0,0.0,0.0,1.0,1.0,1.0,0.0,100.0,1.0,0.0,5.0,0.0,2005,NaN
3,200500000003,2.0,41.0,0.0,NaN,0.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,0.0,2.0,0.0,5.0,0.0,2005,NaN
4,200500000004,3.0,916.0,0.0,NaN,2.0,2.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,2005,NaN
5,200500000005,3.0,110.0,0.0,NaN,2.0,2.0,24.0,630.0,0.0,1.0,3.0,0.0,59.0,2.0,0.0,3.0,0.0,2005,NaN


In [100]:
df_lieux.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1017309 entries, 1 to 1017309
Data columns (total 20 columns):
 #   Column   Dtype  
---  ------   -----  
 0   num_acc  int64  
 1   catr     float64
 2   voie     object 
 3   v1       float64
 4   v2       object 
 5   circ     float64
 6   nbv      float64
 7   pr       object 
 8   pr1      object 
 9   vosp     float64
 10  prof     float64
 11  plan     float64
 12  lartpc   float64
 13  larrout  float64
 14  surf     float64
 15  infra    float64
 16  situ     float64
 17  env1     float64
 18  annee    int64  
 19  vma      float64
dtypes: float64(14), int64(2), object(4)
memory usage: 163.0+ MB


In [101]:
# au vue du nombre de valeurs manquantes, nous pouvons supprimer : voie, v1, v2, pr, pr1, env1, vma
colonnes_a_supprimer = ['voie','v1', 'v2', 'pr', 'pr1', 'env1', 'vma']
df_lieux = df_lieux.drop(columns=colonnes_a_supprimer)

In [102]:
# uniformisation du type de données parmi tous les dataframes
a_convertir = ['catr', 'circ', 'nbv', 'vosp', 'prof', 'plan', 'lartpc', 'larrout', 'surf', 'infra', 'situ']
df_lieux[a_convertir] = df_lieux[a_convertir].fillna(0).astype('int64')

In [103]:
for item in ['catr', 'circ', 'vosp', 'prof', 'plan', 'surf', 'infra', 'situ']:
    valeurs_uniques = df_lieux[item].unique()
    print(f'Les valeurs prises par {item} sont : {valeurs_uniques}')

Les valeurs prises par catr sont : [3 2 4 6 9 5 1 0 7]
Les valeurs prises par circ sont : [ 2  0  3  4  1 -1]
Les valeurs prises par vosp sont : [ 0  1  3  2 -1]
Les valeurs prises par prof sont : [ 1  0  2  3  4 -1]
Les valeurs prises par plan sont : [ 1  3  2  0  4 -1]
Les valeurs prises par surf sont : [ 1  2  0  9  7  8  5  6  3  4 -1]
Les valeurs prises par infra sont : [ 0  5  4  2  3  6  1  7  9  8 -1]
Les valeurs prises par situ sont : [ 1  5  3  4  0  2  6  8 -1]


Pour le reste du projet, on considère que 0, -1, nan sont équivalents et correspondent à des valeurs manquantes car sans
objet ou non renseignées

In [104]:
# afin de permettre la fusion ultérieure des dataframes, on supprime la colonne commune 'annee' que nous laisserons dans df_usagers
df_lieux = df_lieux.drop(columns=['annee'])

In [105]:
# quelques visualisations dans le cadre de l analyse des données manipulées

# représentation du nombre d accidents en fonction du type de routes

effectifs = df_lieux['catr'].value_counts().reset_index()
effectifs.columns = ['Catégorie de Route', 'Nombre d\'Accidents']


fig = px.bar(effectifs, x='Catégorie de Route', y='Nombre d\'Accidents',
    title='Effectif des Accidents en Fonction du Type de Route',
    labels={'Catégorie de Route': 'Catégorie de Route', 'Nombre d\'Accidents': 'Nombre d\'Accidents'},
    color='Nombre d\'Accidents',
    color_continuous_scale='Blues')
fig.show()

In [61]:
# quelques visualisations dans le cadre de l analyse des données manipulées

# représentation du nombre d accidents en fonction de l'état de surface des routes

effectifs = df_lieux['surf'].value_counts().reset_index()
effectifs.columns = ['Etat de surface de la route', 'Nombre d\'Accidents']


fig = px.bar(effectifs, x='Etat de surface de la route', y='Nombre d\'Accidents',
        title='Effectif des Accidents en Fonction de l\'État de Surface des Routes',
        labels={'Etat de surface de la route': 'État de surface', 'Nombre d\'Accidents': 'Nombre d\'Accidents'},
        color='Nombre d\'Accidents',  
        color_continuous_scale='Blues')

fig.update_layout(
        xaxis=dict(
            tickmode='array',
            tickvals=effectifs['Etat de surface de la route'],
            ticktext=effectifs['Etat de surface de la route'],
            automargin=True  
        )
    )
fig.show()

### 1.2 Données 'caractéristiques'

In [106]:
df_carac = pd.read_csv('caracteristiques_2005a2021.csv', encoding='iso-8859-1', index_col=0)
df_carac.head()

,num_acc,mois,jour,hrmn,lum,agg,int,atm,col,com,adr,gps,lat,long,dep,annee
1,200500000001,1,12,1900,3,2,1,1.0,3.0,11,CD41B,M,5051500.0,294400.0,590,2005
2,200500000002,1,21,1600,1,2,1,1.0,1.0,51,rue de Lille,M,5053700.0,280200.0,590,2005
3,200500000003,1,21,1845,3,1,1,2.0,1.0,51,NaN,M,5054600.0,280000.0,590,2005
4,200500000004,1,4,1615,1,1,1,1.0,5.0,82,NaN,M,5098700.0,240800.0,590,2005
5,200500000005,1,10,1945,3,1,1,3.0,6.0,478,NaN,M,5096400.0,247500.0,590,2005


In [107]:
df_carac.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1121571 entries, 1 to 1121571
Data columns (total 16 columns):
 #   Column   Dtype  
---  ------   -----  
 0   num_acc  int64  
 1   mois     int64  
 2   jour     int64  
 3   hrmn     object 
 4   lum      int64  
 5   agg      int64  
 6   int      int64  
 7   atm      float64
 8   col      float64
 9   com      object 
 10  adr      object 
 11  gps      object 
 12  lat      object 
 13  long     object 
 14  dep      object 
 15  annee    int64  
dtypes: float64(2), int64(7), object(7)
memory usage: 145.5+ MB


In [108]:
!pip install notebook plotly


[notice] A new release of pip is available: 24.0 -> 24.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [13]:
# Initialiser Plotly pour s'assurer qu'il fonctionne bien dans Jupyter Notebook
init_notebook_mode(connected=True)

In [14]:
# On regarde si les latitudes et longitudes font ressortir des insights particuliers
# Si non, au vue du nombre de valeurs manquantes, nous supprimerons ces colonnes

df_carac['lat'] = pd.to_numeric(df_carac['lat'], errors='coerce')
df_carac['long'] = pd.to_numeric(df_carac['long'], errors='coerce')

df_carac.dropna(subset=['lat', 'long'], inplace=True)

fig = px.scatter_mapbox(df_carac, lat='lat', lon='long', zoom=5, height=600)

fig.update_layout(mapbox_style="open-street-map")
fig.update_layout(margin={"r":0,"t":0,"l":0,"b":0})

fig.show()

In [109]:
# au vue du nombre de valeurs manquantes, nous pouvons supprimer : adr, gps, lat, long
colonnes_a_supprimer = ['adr','gps', 'lat', 'long']
df_carac = df_carac.drop(columns=colonnes_a_supprimer)

In [110]:
# on encode les variables le nécessitant
df_carac['dep'] = df_carac['dep'].replace({'2A': 1000, '2B': 1001})

In [111]:
# on supprime les données qui ne sont pas entières (bien que leur format laisse à penser que 'dep' et 'com' ont été fusionnés)
def filter_non_int(com):
    try:
        int(com)
        return com
    except ValueError:
        return None

df_carac = df_carac.applymap(filter_non_int)

C:\Users\Administrateur\AppData\Local\Temp\ipykernel_10680\3490412005.py:9: FutureWarning:

DataFrame.applymap has been deprecated. Use DataFrame.map instead.



In [112]:
# on distingue les heures et les minutes
df_carac[['hr', 'mn']] = df_carac['hrmn'].str.extract(r'(\d{2})(\d{2})')
df_carac.drop('hrmn', axis=1, inplace=True)

In [113]:
# uniformisation du type de données parmi tous les dataframes
a_convertir = ['atm', 'col','dep', 'com', 'hr', 'mn' ]
df_carac[a_convertir] = df_carac[a_convertir].fillna(0).astype('int64')

In [114]:
df_carac.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1121571 entries, 1 to 1121571
Data columns (total 13 columns):
 #   Column   Dtype
---  ------   -----
 0   num_acc  int64
 1   mois     int64
 2   jour     int64
 3   lum      int64
 4   agg      int64
 5   int      int64
 6   atm      int64
 7   col      int64
 8   com      int64
 9   dep      int64
 10  annee    int64
 11  hr       int64
 12  mn       int64
dtypes: int64(13)
memory usage: 119.8 MB


In [115]:
for item in ['lum', 'agg', 'int', 'atm', 'col', 'jour', 'mois']:
    valeurs_uniques = df_carac[item].unique()
    print(f'Les valeurs prises par {item} sont : {valeurs_uniques}')

Les valeurs prises par lum sont : [ 3  1  5  4  2 -1]
Les valeurs prises par agg sont : [2 1]
Les valeurs prises par int sont : [ 1  2  9  0  6  8  3  4  7  5 -1]
Les valeurs prises par atm sont : [ 1  2  3  8  9  7  6  4  5  0 -1]
Les valeurs prises par col sont : [ 3  1  5  6  2  4  7  0 -1]
Les valeurs prises par jour sont : [12 21  4 10 28  3 18 25 29 23 11  1 30 19  9 31 15 13 26  2  6  8 20 16
  7  5 24 14 17 22 27]
Les valeurs prises par mois sont : [ 1  2  3  4  5  6  7  8  9 10 11 12]


Remarques
- dep : code insee du département
- com : code composé du code INSEE du département (ci dessus) suivi par 3 chiffres - il n'y en a que 2 

In [72]:
# afin de permettre la fusion ultérieure des dataframes, on supprime la colonne commune 'annee' que nous laisserons dans df_usagers
df_carac = df_carac.drop(columns=['annee'])

### 1.3 Données 'usagers'

In [116]:
df_usagers=pd.read_csv ('usagers_2005a2021.csv', index_col=0)
df_usagers.head()

,num_acc,place,catu,grav,sexe,trajet,secu,locp,actp,etatp,an_nais,num_veh,annee,id_vehicule,secu1,secu2,secu3
1,2.005000e+11,1.0,1,4,1,1.0,11.0,0.0,0,0.0,1976.0,A01,2005,NaN,NaN,NaN,NaN
2,2.005000e+11,1.0,1,3,2,3.0,11.0,0.0,0,0.0,1968.0,B02,2005,NaN,NaN,NaN,NaN
3,2.005000e+11,2.0,2,1,1,0.0,11.0,0.0,0,0.0,1964.0,B02,2005,NaN,NaN,NaN,NaN
4,2.005000e+11,4.0,2,1,1,0.0,31.0,0.0,0,0.0,2004.0,B02,2005,NaN,NaN,NaN,NaN
5,2.005000e+11,5.0,2,1,1,0.0,11.0,0.0,0,0.0,1998.0,B02,2005,NaN,NaN,NaN,NaN


In [117]:
df_usagers.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2509620 entries, 1 to 2509620
Data columns (total 17 columns):
 #   Column       Dtype  
---  ------       -----  
 0   num_acc      float64
 1   place        float64
 2   catu         int64  
 3   grav         int64  
 4   sexe         int64  
 5   trajet       float64
 6   secu         float64
 7   locp         float64
 8   actp         object 
 9   etatp        float64
 10  an_nais      float64
 11  num_veh      object 
 12  annee        int64  
 13  id_vehicule  object 
 14  secu1        float64
 15  secu2        float64
 16  secu3        float64
dtypes: float64(10), int64(4), object(3)
memory usage: 344.6+ MB


In [118]:
df_usagers.isna().sum() / len(df_usagers) * 100

num_acc         0.000000
place           4.906241
catu            0.000000
grav            0.000000
sexe            0.000000
trajet          0.019684
secu           16.893474
locp            2.245798
actp            2.249823
etatp           2.248069
an_nais         0.218559
num_veh         0.000000
annee           0.000000
id_vehicule    85.359337
secu1          85.359337
secu2          85.359337
secu3          85.359337
dtype: float64

In [119]:
# au vue du nombre de valeurs manquantes, nous pouvons supprimer : 
colonnes_a_supprimer = ["id_vehicule", "secu1", "secu2", "secu3", "secu"]
df_usagers = df_usagers.drop(columns=colonnes_a_supprimer)

In [120]:
# on encode les variables le nécessitant
df_usagers['actp'] = df_usagers['actp'].replace({'A': 10, 'B': -1})

In [121]:
# uniformisation du type de données parmi tous les dataframes
a_convertir = ['num_acc', 'place', 'trajet', 'locp', 'actp', 'etatp', 'an_nais'] 
df_usagers [a_convertir] = df_usagers [a_convertir].fillna(0).astype('int64')

In [122]:
for item in ['catu', 'grav', 'sexe', 'trajet','locp', 'actp', 'etatp']:
    valeurs_uniques = df_usagers[item].unique()
    print(f'Les valeurs prises par {item} sont : {valeurs_uniques}')

Les valeurs prises par catu sont : [1 2 3 4]
Les valeurs prises par grav sont : [ 4  3  1  2 -1]
Les valeurs prises par sexe sont : [ 1  2 -1]
Les valeurs prises par trajet sont : [ 1  3  0  5  9  4  2 -1]
Les valeurs prises par locp sont : [ 0  2  4  1  5  6  3  8  7 -1  9]
Les valeurs prises par actp sont : [ 0  3  1  5  2  9  6  4 -1 10  8  7]
Les valeurs prises par etatp sont : [ 0  2  1  3 -1]


In [123]:
# les colonnes catu ne peuvent pas intégrer la valeur '4'
# les colonnes sans la valeur cible du projet 'grav' renseignée n'ont aucune utilité, nous les supprimons donc
condition = (df_usagers['catu'] == 4) | (df_usagers['grav'] == -1)
df_usagers.drop(df_usagers[condition].index, inplace=True)

NOTA 1
 - L’indicateur « blessé hospitalisé » n’est plus labellisé par l’autorité de la statistique publique depuis 2019.
 Il nous faut donc le changer.

 NOTA 2
 - A partir des données de 2021, les usagers en fuite ont été rajoutés, cela entraîne des manques d’informations sur ces derniers, notamment le sexe, l’âge, voire la gravité des blessures (indemne, blessé léger ou blessé hospitalisé)
 Notre analyse s'arrêtant en 2021, cela n aura - pour notre projet - pas d'impact.

In [124]:
# on rassemble les blessés de type 4 avec les personnes indemne
df_usagers['grav'] = df_usagers['grav'].replace({4 : 1})

In [82]:
# quelques visualisations dans le cadre de l analyse des données manipulées

# représentation du nombre d accidents en fonction du type de routes

effectifs = df_usagers['grav'].value_counts().reset_index()
effectifs.columns = ['Gravité accident', 'Nombre de blessés']

effectifs['Gravité accident'] = effectifs['Gravité accident'].astype(str)
category_order = ['1', '2', '3']

fig = px.bar(effectifs, x='Gravité accident', y='Nombre de blessés',
    title='Effectif des Accidents en Fonction de leur gravité',
    labels={'Gravité accident': 'Gravité accident', 'Nombre de blessés': 'Nombre de blessés'},
    color='Nombre de blessés',
    color_continuous_scale='Blues',
    category_orders={'Gravité accident': category_order}
)

fig.update_xaxes(tickvals=category_order)

fig.show()

In [83]:
# quelques visualisations dans le cadre de l analyse des données manipulées

# représentation du nombre d accidents en fonction du type de trajet

effectifs = df_usagers['trajet'].value_counts().reset_index()
effectifs.columns = ['Type de trajet', 'Nombre de blessés']

effectifs['Type de trajet'] = effectifs['Type de trajet'].astype(str)

category_order = ['-1', '1', '2', '3', '4', '5', '9']

fig = px.bar(effectifs, x='Type de trajet', y='Nombre de blessés',
    title='Effectif des Accidents en Fonction du Type de trajet',
    labels={'Type de trajet': 'Type de trajet', 'Nombre de blessés': 'Nombre de blessés'},
    color='Nombre de blessés',
    color_continuous_scale='Blues'
            )

fig.show()

### 1.4 Données 'vehicules'

In [125]:
df_vehicules=pd.read_csv ('vehicules_2005a2021.csv', index_col=0)
df_vehicules.head()

,num_acc,senc,catv,occutc,obs,obsm,choc,manv,num_veh,annee,id_vehicule,motor
1,200500000001,0.0,7,0.0,0.0,2.0,1.0,1.0,A01,2005,NaN,NaN
2,200500000001,0.0,7,0.0,0.0,2.0,8.0,10.0,B02,2005,NaN,NaN
3,200500000002,0.0,7,0.0,0.0,2.0,7.0,16.0,A01,2005,NaN,NaN
4,200500000002,0.0,2,0.0,0.0,2.0,1.0,1.0,B02,2005,NaN,NaN
5,200500000003,0.0,2,0.0,0.0,2.0,1.0,1.0,A01,2005,NaN,NaN


In [126]:
df_vehicules.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1914902 entries, 1 to 1914902
Data columns (total 12 columns):
 #   Column       Dtype  
---  ------       -----  
 0   num_acc      int64  
 1   senc         float64
 2   catv         int64  
 3   occutc       float64
 4   obs          float64
 5   obsm         float64
 6   choc         float64
 7   manv         float64
 8   num_veh      object 
 9   annee        int64  
 10  id_vehicule  object 
 11  motor        float64
dtypes: float64(7), int64(3), object(2)
memory usage: 189.9+ MB


In [127]:
df_vehicules.isna().sum() / len(df_vehicules) * 100

num_acc         0.000000
senc            0.014204
catv            0.000000
occutc         14.456823
obs             0.052535
obsm            0.040629
choc            0.020732
manv            0.024440
num_veh         0.000000
annee           0.000000
id_vehicule    85.425312
motor          85.425312
dtype: float64

In [128]:
# au vue du nombre de valeurs manquantes, nous pouvons supprimer : 
colonnes_a_supprimer = ['occutc', 'id_vehicule', 'motor']
df_vehicules.drop(columns=colonnes_a_supprimer, inplace=True)

In [129]:
# uniformisation du type de données parmi tous les dataframes
a_convertir = ['senc', 'obs', 'obsm', 'choc', 'manv' ]
df_vehicules[a_convertir] = df_vehicules [a_convertir].fillna(0).astype('int64')

In [130]:
for item in ['senc', 'catv', 'obs', 'obsm', 'choc', 'manv']:
    valeurs_uniques = df_vehicules[item].unique()
    valeurs_uniques = sorted(valeurs_uniques)
    print(f'Les valeurs prises par {item} sont : {valeurs_uniques}')

Les valeurs prises par senc sont : [-1, 0, 1, 2, 3]
Les valeurs prises par catv sont : [-1, 0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 50, 60, 80, 99]
Les valeurs prises par obs sont : [-1, 0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17]
Les valeurs prises par obsm sont : [-1, 0, 1, 2, 4, 5, 6, 9]
Les valeurs prises par choc sont : [-1, 0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
Les valeurs prises par manv sont : [-1, 0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26]


In [131]:
# afin de permettre la fusion ultérieure des dataframes, on supprime la colonne commune 'annee' que nous laisserons dans df_usagers
df_vehicules = df_vehicules.drop(columns=['annee'])

### 1.5 Fusion des dataframes

In [132]:
# Jonction de df_usagers avec df_vehicules
df = pd.merge(df_usagers, df_vehicules, on='num_acc', how='inner')

# Jonction du résultat avec df_carac
df = pd.merge(df, df_carac, on='num_acc', how='inner')

# Jonction du résultat avec df_lieux
df = pd.merge(df, df_lieux, on='num_acc', how='inner')

In [133]:
# on affiche l integralite des colonnes pour tout voir
pd.set_option('display.max_info_rows', len(df.columns))

In [93]:
df.head()

,num_acc,place,catu,grav,sexe,trajet,locp,actp,etatp,an_nais,...,circ,nbv,vosp,prof,plan,lartpc,larrout,surf,infra,situ
0,200500000001,1,1,1,1,1,0,0,0,1976,...,2,2,0,1,1,0,63,1,0,1
1,200500000001,1,1,1,1,1,0,0,0,1976,...,2,2,0,1,1,0,63,1,0,1
2,200500000001,1,1,3,2,3,0,0,0,1968,...,2,2,0,1,1,0,63,1,0,1
3,200500000001,1,1,3,2,3,0,0,0,1968,...,2,2,0,1,1,0,63,1,0,1
4,200500000001,2,2,1,1,0,0,0,0,1964,...,2,2,0,1,1,0,63,1,0,1


In [134]:
# on transforme les -1 et 0 en NaN (signification identique)

# colonne non catégorielle
colonne_Non_cat=['num_acc', 'an_nais', "num_veh_x", 'annee', "num_veh_y", 'mois', 'jour', 'com', 'dep', 'hr', 'mn','nbv','lartpc','larrout']

# colonne catégorielle
cat_columns = [col for col in df.columns if col not in colonne_Non_cat]

df[cat_columns] = df[cat_columns].replace({-1: np.nan})

In [97]:
df.isna().sum() / len(df) * 100

num_acc       0.000000
place         4.572027
catu          0.000000
grav          0.000000
sexe          0.000000
trajet       28.923918
locp         95.912740
actp         95.689590
etatp        95.710430
an_nais       0.000000
num_veh_x     0.000000
annee         0.000000
senc         81.937215
catv          0.003815
obs          89.497269
obsm         17.721649
choc          5.538326
manv          7.624142
num_veh_y     0.000000
mois          0.000000
jour          0.000000
lum           0.000000
agg           0.000000
int           0.012074
atm           0.006375
col           0.002342
com           0.000000
dep           0.000000
hr            0.000000
mn            0.000000
catr          0.000048
circ          4.654009
nbv           0.000000
vosp         93.955222
prof          6.665836
plan          6.415085
lartpc        0.000000
larrout       0.000000
surf          2.850408
infra        88.190413
situ          4.718145
dtype: float64

In [135]:
# au vue du nombre de valeurs manquantes, nous pouvons supprimer : 
colonnes_a_supprimer = ['locp','actp', 'etatp', 'senc', 'obs', 'obsm', 'vosp', 'infra']
df = df.drop(columns=colonnes_a_supprimer)

In [136]:
# on supprime désormais les lignes avec des NaN (ne pouvant les remplacer par d'autres catégories de façon aléatoire)
df_cleaned = df.dropna()

In [137]:
df_cleaned.isna().sum() / len(df_cleaned) * 100

num_acc      0.0
place        0.0
catu         0.0
grav         0.0
sexe         0.0
trajet       0.0
an_nais      0.0
num_veh_x    0.0
annee_x      0.0
catv         0.0
choc         0.0
manv         0.0
num_veh_y    0.0
mois         0.0
jour         0.0
lum          0.0
agg          0.0
int          0.0
atm          0.0
col          0.0
com          0.0
dep          0.0
annee_y      0.0
hr           0.0
mn           0.0
catr         0.0
circ         0.0
nbv          0.0
prof         0.0
plan         0.0
lartpc       0.0
larrout      0.0
surf         0.0
situ         0.0
dtype: float64

In [139]:
df_cleaned.info() # comment pouvons nous avoir 4M de lignes là ou tous les jeux de données n en ont que 1M ??

<class 'pandas.core.frame.DataFrame'>
Index: 4136759 entries, 0 to 4141160
Data columns (total 34 columns):
 #   Column     Dtype  
---  ------     -----  
 0   num_acc    int64  
 1   place      int64  
 2   catu       int64  
 3   grav       int64  
 4   sexe       int64  
 5   trajet     float64
 6   an_nais    int64  
 7   num_veh_x  object 
 8   annee_x    int64  
 9   catv       int64  
 10  choc       float64
 11  manv       float64
 12  num_veh_y  object 
 13  mois       int64  
 14  jour       int64  
 15  lum        int64  
 16  agg        int64  
 17  int        int64  
 18  atm        float64
 19  col        float64
 20  com        int64  
 21  dep        int64  
 22  annee_y    int64  
 23  hr         int64  
 24  mn         int64  
 25  catr       int64  
 26  circ       float64
 27  nbv        int64  
 28  prof       float64
 29  plan       float64
 30  lartpc     int64  
 31  larrout    int64  
 32  surf       float64
 33  situ       float64
dtypes: float64(10), int64(2

In [138]:
# on crée un .csv du jeu de données après nettoyage
df_cleaned.to_csv('data 2005a2021.csv', index=False)

In [46]:
# identifier le lieu de stockage du document 
"""repertoire_courant = os.getcwd()
print(repertoire_courant)"""